# 02 — Segment Trials

**Purpose:** Split a continuous recording into per-trial CSV files using Pupil Neon annotations.

## Pipeline

1. Set `DATA_DIR` to a Pupil Neon export that includes `imu.csv`, `gaze_positions.csv`, `blinks.csv`, and `annotations.csv`.
2. Compute gaze angle (same as Notebook 01).
3. Optionally mask blink intervals with NaN.
4. Parse trial start/end annotations and segment the data.
5. Save one CSV per trial to `OUTPUT_DIR`.

## Outputs

One CSV per trial in `OUTPUT_DIR`, named like:

    Subject-03_Hill-Condition_Trial-01_Walk-Dir-Up.csv

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from neon_gaze.io import load_imu, load_gaze_positions, load_blinks, load_annotations
from neon_gaze.processing import synchronize_gaze_to_imu, compute_gaze_angle, mask_blinks
from neon_gaze.segmentation import add_trial_events, segment_and_save_trials
from neon_gaze.plotting import plot_gaze_angle

## Configuration

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# Path to a Pupil Neon export that includes annotations
DATA_DIR = "/path/to/your/neon_player/exports/000_session"

# Subject/condition label used in output filenames
SUBJECT_ID = "Subject-03_Hill-Condition"

# Output directory for segmented trial CSVs
OUTPUT_DIR = "../data/walk_segmented_csvs"

# Number of expected trials
EXPECTED_TRIALS = 20

# Set to True to actually save the trial CSVs
SAVE_OUTPUT = False

## Step 1 — Load all data streams

In [ ]:
imu_df = load_imu(os.path.join(DATA_DIR, "imu.csv"))
gaze_df = load_gaze_positions(os.path.join(DATA_DIR, "gaze_positions.csv"))
blinks_df = load_blinks(os.path.join(DATA_DIR, "blinks.csv"))
annotations_df = load_annotations(os.path.join(DATA_DIR, "annotations.csv"))

print(f"IMU rows:         {len(imu_df)}")
print(f"Gaze rows:        {len(gaze_df)}")
print(f"Blinks:           {len(blinks_df)}")
print(f"Annotations:      {len(annotations_df)}")

## Step 2 — Compute gaze angle

In [ ]:
merged_df = synchronize_gaze_to_imu(imu_df, gaze_df)
gaze_angle_df = compute_gaze_angle(merged_df)

print(f"Gaze angle rows (no blink mask): {len(gaze_angle_df)}")
display(gaze_angle_df.head())

## Step 3 — Mask blinks

Blink intervals are set to NaN so they do not contaminate downstream analyses.

In [ ]:
gaze_angle_blinked_df = mask_blinks(gaze_angle_df, blinks_df)
display(gaze_angle_blinked_df.head())

## Step 4 — Add trial events from annotations

In [ ]:
gaze_angle_blinked_df = add_trial_events(
    gaze_angle_blinked_df, annotations_df, expected_trials=EXPECTED_TRIALS
)

# Show rows where trial events were marked
display(gaze_angle_blinked_df[gaze_angle_blinked_df["trial event"].notna()].head(10))

## Step 5 — Segment and save

In [ ]:
if SAVE_OUTPUT:
    saved_paths = segment_and_save_trials(
        gaze_angle_blinked_df,
        output_dir=OUTPUT_DIR,
        subject_id=SUBJECT_ID,
        expected_trials=EXPECTED_TRIALS,
    )
    print(f"\nSaved {len(saved_paths)} trial files.")
else:
    print("SAVE_OUTPUT is False — set to True in the config cell to save.")

## Visual inspection

Use the dropdown to browse individual trial segments.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown
from IPython.display import display, clear_output

csv_files = sorted(
    f for f in os.listdir(OUTPUT_DIR) if f.endswith(".csv")
) if os.path.isdir(OUTPUT_DIR) else []

if csv_files:
    dropdown = Dropdown(
        options=csv_files,
        value=csv_files[0],
        description="Segment:",
        style={"description_width": "initial"},
        layout={"width": "500px"},
    )

    def plot_segment(change):
        if change["name"] != "value" or change["new"] is None:
            return
        filename = change["new"]
        clear_output(wait=True)
        display(dropdown)
        df = pd.read_csv(os.path.join(OUTPUT_DIR, filename))
        plt.figure(figsize=(10, 4))
        plt.plot(df["time_sec"], df["gaze angle [deg]"])
        plt.xlabel("Time (s)")
        plt.ylabel("Gaze angle (deg)")
        plt.title(f"Gaze angle: {filename}")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    dropdown.observe(plot_segment, names="value")
    display(dropdown)
    plot_segment({"name": "value", "new": dropdown.value})
else:
    print(f"No segmented CSVs found in '{OUTPUT_DIR}'. Run Step 5 first.")